# Notebook 3 -- Predicting the LLMs' votes from dialogue features

**Task.** For each (LLM, game): which roster member does the LLM vote for?
Framed as a per-game ranking -- every roster member gets a score, the argmax is
the prediction, and the metric is **top-1 accuracy per game**. This differs
from Lai et al.'s pairwise task (did A vote B, F1 0.33), so numbers are NOT
comparable across the two setups.

**Labels.** Two label sources, never pooled: `stochastic` uses each of the 3
T=1 run votes as its own labeled instance (all instances of a game stay in the
same CV fold), so split games are kept and the reported metric is the expected
probability that a single T=1 sample matches the prediction; `greedy` uses the
single deterministic vote. Circle-vote instances have no roster candidate and
are excluded per label source (counts reported).

**Feature families** (nested by design):
- **A_pt** -- human-annotated persuasion techniques, speaker perspective:
  per-player counts of Accusation, Defense, Interrogation, Identity
  Declaration, Evidence, Call for Action, plus utterance count.
- **B_directed** -- the DeepSeek-enriched layer: werewolf- and deception-type
  accusations *received*, self-claimed Werewolf, role-claim conflicts,
  self-contradictions.
- **C_combined** -- A + B.

**Baselines** (also findings): uniform random, most-talkative, most-accused
(werewolf-type), and **crowd-modal** -- predict the human village modal target.
Crowd-modal is deliberately a baseline and not a feature: human votes are
almost never declared in the transcript, so the LLM cannot see them; matching
this baseline measures whether LLMs and humans *interpret games similarly*,
nothing causal.

**Evaluation.** GroupKFold by game (5 folds x 3 shuffle seeds), mean +/- std.
Interpretation uses **held-out permutation importance** (per fold, custom
top-1 metric) for the best family/method per (LLM, label source), plus
standardized logreg coefficients from a full-data fit for reference only.


## Config and paths

In [1]:
from pathlib import Path
from typing import Optional
from collections import Counter
from itertools import combinations
import json
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

REPO_NAME = "masters_thesis_sdg"
MODEL_STAGE = "base"
PROMPT_DIR = "prompt_v4"
STOCHASTIC_RUNS = ["run_1", "run_2", "run_3"]
GREEDY_RUN = "greedy_t0"
N_FOLDS = 5
CV_SEEDS = [0, 1, 2]
RANDOM_STATE = 42

PT_LABELS = ["Accusation", "Defense", "Interrogation", "Identity Declaration",
             "Evidence", "Call for Action"]

FAMILY_A = [f"pt_{l.lower().replace(' ', '_')}" for l in PT_LABELS] + ["n_utterances"]
FAMILY_B = ["werewolf_count", "deception_count", "claims_werewolf",
            "made_any_claim", "claims_info_role", "n_distinct_roles_claimed_self",
            "is_in_role_conflict", "is_self_contradiction"]
# Provenance families (did the DeepSeek enrichment add value?)
FAMILIES = {"A_pt": FAMILY_A, "B_directed": FAMILY_B, "C_combined": FAMILY_A + FAMILY_B}
# Mechanism families (the RQ contrast): social-pressure signals vs
# consistency/deduction signals. claims_werewolf (taking a self-declaration
# literally) is grouped with the deductive/content side -- move it if you
# disagree; accusation COUNTS are social salience (the content of an
# accusation may carry evidence, but the count does not).
FAMILY_SOCIAL = FAMILY_A + ["werewolf_count", "deception_count"]
FAMILY_DEDUCTIVE = ["is_in_role_conflict", "is_self_contradiction", "claims_werewolf",
                    "made_any_claim", "claims_info_role", "n_distinct_roles_claimed_self"]
FAMILIES.update({"S_social": FAMILY_SOCIAL, "D_deductive": FAMILY_DEDUCTIVE})
METHODS = ["logreg", "gbm", "lasso"]


def find_repo_root(start=None, repo_name=REPO_NAME):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(repo_name)
        current = current.parent


REPO_ROOT = find_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "analysis"
TABLES_REL = Path(MODEL_STAGE) / "voting" / PROMPT_DIR / "vote_stability" / "tables"
ANNOT_ROOT = REPO_ROOT / "data" / "lai2023"
ACC_TARGETS_ROOT = REPO_ROOT / "data" / "processed" / "lai2023" / "accusation_transcripts" / "acc_targets"
IC_FEATURES_CSV = (REPO_ROOT / "data" / "processed" / "lai2023"
                   / "identity_claim_transcripts" / "ic_targets" / "player_conflict_features.csv")
HUMAN_TABLES_DIR = ANALYSIS_ROOT / "human_outcomes" / PROMPT_DIR / "tables"
OUTPUT_DIR = ANALYSIS_ROOT / "cross_model" / "voting" / PROMPT_DIR / "predictive" / "tables"
print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg


## Canonical game keys

Annotations, DeepSeek outputs, and the vote tables spell sessions differently
(`#` vs spaces, case). Everything is joined on a canonical key.


In [2]:
def canonical_session(x):
    x = str(x).strip().replace("#", " ")
    return re.sub(r"\s+", " ", x).lower()


def canonical_game(x):
    m = re.search(r"\d+", str(x))
    return f"game{int(m.group())}" if m else str(x).strip().lower()


def ckey(source, session, game):
    return (str(source).strip(), canonical_session(session), canonical_game(game))


## Labels and rosters (from notebook 1's tables)

One labeled instance per (LLM, game, run) with a named vote. Rosters come from
the game-level table so silent players still appear as candidates.


In [3]:
def short_model_label(name):
    m = re.search(r"(\d+B)", name)
    return m.group(1) if m else name


file_frames, game_frames = [], []
for model_dir in sorted(ANALYSIS_ROOT.iterdir()):
    tables = model_dir / TABLES_REL
    if not (tables / "llm_vote_file_level.csv").exists():
        continue
    label = short_model_label(model_dir.name)
    f = pd.read_csv(tables / "llm_vote_file_level.csv"); f["model"] = label
    g = pd.read_csv(tables / "llm_vote_game_level.csv"); g["model"] = label
    file_frames.append(f); game_frames.append(g)
votes = pd.concat(file_frames, ignore_index=True)
games = pd.concat(game_frames, ignore_index=True)
MODELS = sorted(votes["model"].unique())
print("Models:", MODELS)

roster_by_key = {}
for _, r in games.drop_duplicates(subset=["source", "session_name", "game_key"]).iterrows():
    roster_by_key[ckey(r["source"], r["session_name"], r["game_key"])] = {
        "source": r["source"], "session": r["session_name"], "game": r["game_key"],
        "players": json.loads(r["player_names"]) if isinstance(r["player_names"], str) else r["player_names"],
    }
print("games with rosters:", len(roster_by_key))

instance_rows, excluded = [], Counter()
for _, r in votes.iterrows():
    label_source = "greedy" if r["run_label"] == GREEDY_RUN else "stochastic"
    if r["run_label"] not in STOCHASTIC_RUNS and r["run_label"] != GREEDY_RUN:
        continue
    if r["status"] not in ("player_vote", "circle_vote"):
        excluded[(label_source, r["status"])] += 1
        continue
    if bool(r["is_circle_vote"]):
        excluded[(label_source, "circle_vote")] += 1
        continue
    instance_rows.append({"model": r["model"], "label_source": label_source,
                          "key": ckey(r["source"], r["session_name"], r["game_key"]),
                          "run_label": r["run_label"], "voted": r["chosen_player_name"]})
instances = pd.DataFrame(instance_rows)
print("labeled instances:", instances.groupby(["model", "label_source"]).size().to_dict())
print("excluded (no roster candidate):", dict(excluded))


Models: ['2B', '31B', '4B']
games with rosters: 191
labeled instances: {('2B', 'greedy'): 143, ('2B', 'stochastic'): 437, ('31B', 'greedy'): 167, ('31B', 'stochastic'): 504, ('4B', 'greedy'): 173, ('4B', 'stochastic'): 539}
excluded (no roster candidate): {('stochastic', 'circle_vote'): 234, ('greedy', 'circle_vote'): 90, ('stochastic', 'failed_parse'): 5}


## Family A: persuasion-technique counts (human annotations)

All splits of both sources are pooled -- splits are rebuilt as k-folds below.
The loader reports the annotation-to-roster match rate; investigate any
unmatched games before trusting downstream numbers.


In [4]:
def iter_annotation_games(annot_root):
    for split_file in sorted(annot_root.rglob("split/*.json")):
        source = split_file.parent.parent.name
        data = json.loads(split_file.read_text(encoding="utf-8"))
        for game in data:
            session = (game.get("video_name") or game.get("session")
                       or game.get("YT_ID") or game.get("EG_ID"))
            game_id = game.get("Game_ID") or game.get("game")
            if session is None or game_id is None:
                continue
            yield source, session, game_id, game.get("Dialogue", [])


pt_rows, seen_keys = [], set()
for source, session, game_id, dialogue in iter_annotation_games(ANNOT_ROOT):
    key = ckey(source, session, game_id)
    if key in seen_keys:            # same game can appear via duplicate files
        continue
    seen_keys.add(key)
    per_speaker = {}
    for utt in dialogue:
        sp = str(utt.get("speaker", "")).strip()
        if not sp:
            continue
        d = per_speaker.setdefault(sp, Counter())
        d["n_utterances"] += 1
        for ann in utt.get("annotation", []):
            if ann in PT_LABELS:
                d[f"pt_{ann.lower().replace(' ', '_')}"] += 1
    for sp, counts in per_speaker.items():
        pt_rows.append({"key": key, "speaker": sp, **counts})
pt_df = pd.DataFrame(pt_rows).fillna(0)
print(f"annotation games loaded: {len(seen_keys)}")

matched = sum(1 for k in seen_keys if k in roster_by_key)
print(f"annotation games matching a roster key: {matched} / {len(seen_keys)}")
missing = [k for k in roster_by_key if k not in seen_keys]
print(f"roster games with NO annotation match: {len(missing)}")
for k in missing[:5]:
    print("   e.g.", k)


annotation games loaded: 199
annotation games matching a roster key: 191 / 199
roster games with NO annotation match: 0


## Family B: directed accusations + identity-claim features


In [5]:
acc_rows = []
for p in sorted(ACC_TARGETS_ROOT.rglob("*.json")):
    try:
        rec = json.loads(p.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        continue
    meta = rec.get("metadata", {})
    if not all(meta.get(k) for k in ("source", "session", "game")):
        continue
    key = ckey(meta["source"], meta["session"], meta["game"])
    for item in rec.get("items", []):
        for rel in item.get("relations", []):
            if rel.get("type") not in ("werewolf", "deception"):
                continue
            for pl in rel.get("accused", []):
                if pl != "UNKNOWN":
                    acc_rows.append({"key": key, "player": pl, "relation_type": rel["type"]})
acc_df = pd.DataFrame(acc_rows)
acc_counts = (acc_df.groupby(["key", "player", "relation_type"]).size().unstack(fill_value=0)
              .reset_index()) if not acc_df.empty else pd.DataFrame(columns=["key", "player"])
for col, new in [("werewolf", "werewolf_count"), ("deception", "deception_count")]:
    if col in acc_counts.columns:
        acc_counts = acc_counts.rename(columns={col: new})
    else:
        acc_counts[new] = 0
print(f"accusation events: {len(acc_df)}; games covered: {acc_counts['key'].nunique() if len(acc_counts) else 0}")

ic = pd.read_csv(IC_FEATURES_CSV)
ic["key"] = [ckey(s, se, g) for s, se, g in zip(ic["source"], ic["session"], ic["game"])]
INFO_ROLES = {"Seer", "Robber", "Troublemaker", "Insomniac"}
def parse_roles(v):
    try:
        return set(json.loads(v)) if isinstance(v, str) else set()
    except json.JSONDecodeError:
        return set()
ic["roles_set"] = ic["roles_claimed"].apply(parse_roles)
ic["claims_werewolf"] = ic["roles_set"].apply(lambda s: int("Werewolf" in s))
ic["made_any_claim"] = ic["roles_set"].apply(lambda s: int(len(s) > 0))
ic["claims_info_role"] = ic["roles_set"].apply(lambda s: int(bool(s & INFO_ROLES)))
ic["n_distinct_roles_claimed_self"] = pd.to_numeric(
    ic["n_distinct_roles_claimed_self"], errors="coerce").fillna(0)
for col in ("is_in_role_conflict", "is_self_contradiction"):
    ic[col] = ic[col].astype(str).str.lower().eq("true").astype(int)
ic_feats = ic[["key", "player", "claims_werewolf", "made_any_claim", "claims_info_role",
               "n_distinct_roles_claimed_self", "is_in_role_conflict", "is_self_contradiction"]]
print(f"identity-claim rows: {len(ic_feats)}; games covered: {ic_feats['key'].nunique()}")


accusation events: 2001; games covered: 182
identity-claim rows: 632; games covered: 189


## Candidate table

One row per (game, roster player), features merged by name (case-insensitive
against the roster). Missing = 0: a player never annotated as doing or
receiving anything genuinely has zero counts.


In [6]:
def name_map(players):
    return {p.strip().lower(): p for p in players}


ALL_FEATURES = FAMILIES["C_combined"]
cand_rows = []
for key, info in roster_by_key.items():
    nmap = name_map(info["players"])

    def canon_player(name):
        return nmap.get(str(name).strip().lower())

    feats = {p: dict.fromkeys(ALL_FEATURES, 0.0) for p in info["players"]}
    for df, cols in [(pt_df[pt_df["key"] == key], FAMILY_A),
                     (acc_counts[acc_counts["key"] == key] if len(acc_counts) else pd.DataFrame(),
                      ["werewolf_count", "deception_count"]),
                     (ic_feats[ic_feats["key"] == key],
                      ["claims_werewolf", "made_any_claim", "claims_info_role",
                       "n_distinct_roles_claimed_self", "is_in_role_conflict", "is_self_contradiction"])]:
        for _, r in df.iterrows():
            p = canon_player(r.get("speaker", r.get("player")))
            if p is None:
                continue
            for c in cols:
                if c in r and pd.notna(r[c]):
                    feats[p][c] += float(r[c])
    for p, f in feats.items():
        cand_rows.append({"key": key, "source": info["source"], "session": info["session"],
                          "game": info["game"], "player": p, **f})
candidates = pd.DataFrame(cand_rows)
print(f"candidate rows: {len(candidates)} over {candidates['key'].nunique()} games")
print("nonzero feature coverage (share of rows > 0):")
display((candidates[ALL_FEATURES] > 0).mean().round(3).to_frame("share_nonzero"))


candidate rows: 864 over 191 games
nonzero feature coverage (share of rows > 0):


,share_nonzero
pt_accusation,0.854
pt_defense,0.816
pt_interrogation,0.914
pt_identity_declaration,0.728
pt_evidence,0.832
pt_call_for_action,0.631
n_utterances,0.978
werewolf_count,0.527
deception_count,0.416
claims_werewolf,0.110


## Training rows and evaluation machinery

Each labeled instance expands to its game's candidate rows (label = 1 for the
voted player). CV is grouped by game with shuffled fold assignment per seed.
The metric is instance-level top-1, averaged per game first so every game
weighs equally regardless of how many valid runs it has.


In [7]:
def build_training(instances, candidates, model, label_source):
    inst = instances[(instances["model"] == model) & (instances["label_source"] == label_source)]
    rows = []
    for _, r in inst.iterrows():
        cand = candidates[candidates["key"] == r["key"]]
        if cand.empty or r["voted"] not in set(cand["player"]):
            continue
        for _, c in cand.iterrows():
            rows.append({"key": r["key"], "instance": (r["key"], r["run_label"]),
                         "player": c["player"], "label": int(c["player"] == r["voted"]),
                         **{f: c[f] for f in ALL_FEATURES}})
    return pd.DataFrame(rows)


def top1_score(df, score_col):
    # per instance: argmax candidate == voted?  then mean per game, then mean over games
    def inst_ok(g):
        return int(g.loc[g[score_col].idxmax(), "label"] == 1)
    per_inst = df.groupby("instance", sort=False).apply(inst_ok, include_groups=False)
    inst_game = pd.DataFrame({"ok": per_inst,
                              "game": [i[0] for i in per_inst.index]})
    return float(inst_game.groupby("game")["ok"].mean().mean())


def shuffled_folds(keys, n_folds, seed):
    keys = list(keys)
    np.random.default_rng(seed).shuffle(keys)
    return {k: i % n_folds for i, k in enumerate(keys)}


def fit_method(method, X_train, y_train):
    if method == "logreg":
        m = LogisticRegression(max_iter=1000, class_weight="balanced")
    elif method == "gbm":
        m = GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=50,
                                       max_depth=2, subsample=0.8)
    else:
        m = LogisticRegressionCV(Cs=8, cv=3, penalty="l1", solver="liblinear",
                                 class_weight="balanced", max_iter=2000,
                                 random_state=RANDOM_STATE, scoring="neg_log_loss")
    m.fit(X_train, y_train)
    return m


def cv_evaluate(train_df, feature_cols, collect_perm=False):
    results = {m: [] for m in METHODS}
    perm_deltas = {m: {f: [] for f in feature_cols} for m in METHODS}
    for seed in CV_SEEDS:
        fold_of = shuffled_folds(train_df["key"].unique(), N_FOLDS, seed)
        folds = train_df["key"].map(fold_of)
        for fold in range(N_FOLDS):
            tr, te = train_df[folds != fold].copy(), train_df[folds == fold].copy()
            if te.empty or tr["label"].nunique() < 2:
                continue
            scaler = StandardScaler().fit(tr[feature_cols])
            X_tr, X_te = scaler.transform(tr[feature_cols]), scaler.transform(te[feature_cols])
            for method in METHODS:
                model = fit_method(method, X_tr, tr["label"])
                te["score"] = model.predict_proba(X_te)[:, 1]
                results[method].append(top1_score(te, "score"))
                if collect_perm:
                    base = results[method][-1]
                    rng = np.random.default_rng(RANDOM_STATE + fold)
                    for j, f in enumerate(feature_cols):
                        X_perm = X_te.copy()
                        X_perm[:, j] = rng.permutation(X_perm[:, j])
                        te["score_p"] = model.predict_proba(X_perm)[:, 1]
                        perm_deltas[method][f].append(base - top1_score(te, "score_p"))
    summary = {m: (float(np.mean(v)), float(np.std(v))) for m, v in results.items() if v}
    perm = {m: {f: float(np.mean(d)) for f, d in fs.items() if d} for m, fs in perm_deltas.items()}
    return summary, perm


## Baselines

Uniform random (analytic 1/n per game), most-talkative, most-accused
(werewolf-type), and crowd-modal (human village modal target; only defined on
games with >= 2 village voters -- its n is smaller and reported).


In [8]:
village = pd.read_csv(HUMAN_TABLES_DIR / "village_vote_dispersion.csv")
village["key"] = [ckey(*gid.split(" / ")) for gid in village["game_id"]]
crowd_modal = {r["key"]: set(json.loads(r["village_top_target_names"]))
               for _, r in village.iterrows() if r["n_village_aligned_votes"] >= 2}


def baseline_scores(train_df):
    out = {}
    per_game = train_df.groupby("key")
    out["random_uniform"] = float(np.mean([1.0 / g["player"].nunique()
                                           for _, g in per_game]))
    for name, col in [("most_talkative", "n_utterances"), ("most_accused_ww", "werewolf_count")]:
        out[name] = top1_score(train_df.assign(score=train_df[col]
                                               + np.random.default_rng(0).uniform(0, 1e-6, len(train_df))),
                               "score")
    cm = train_df[train_df["key"].isin(crowd_modal)]
    if not cm.empty:
        cm = cm.assign(score=[1.0 if r["player"] in crowd_modal[r["key"]] else 0.0
                              for _, r in cm.iterrows()])
        cm = cm.assign(score=cm["score"] + np.random.default_rng(0).uniform(0, 1e-6, len(cm)))
        out["crowd_modal"] = top1_score(cm, "score")
        out["crowd_modal_n_games"] = cm["key"].nunique()
    return out


## Run everything

In [9]:
cv_rows, perm_rows, base_rows = [], [], []
train_cache = {}
for model in MODELS:
    for label_source in ["stochastic", "greedy"]:
        train_df = build_training(instances, candidates, model, label_source)
        if train_df.empty:
            continue
        train_cache[(model, label_source)] = train_df
        b = baseline_scores(train_df)
        base_rows.append({"model": model, "label_source": label_source,
                          "n_games": train_df["key"].nunique(),
                          "n_instances": train_df["instance"].nunique(), **b})
        for family, cols in FAMILIES.items():
            summary, _ = cv_evaluate(train_df, cols, collect_perm=False)
            for method, (mean, std) in summary.items():
                cv_rows.append({"model": model, "label_source": label_source,
                                "family": family, "method": method,
                                "top1_mean": round(mean, 3), "top1_std": round(std, 3)})

cv_results = pd.DataFrame(cv_rows)
baselines = pd.DataFrame(base_rows)
print("Baselines:")
display(baselines.round(3))
print("CV results (top-1, mean +/- std over 5 folds x 3 seeds):")
display(cv_results.pivot_table(index=["model", "label_source"],
                               columns=["family", "method"], values="top1_mean").round(3))


Baselines:


,model,label_source,n_games,n_instances,random_uniform,most_talkative,most_accused_ww,crowd_modal,crowd_modal_n_games
0,2B,stochastic,173,437,0.226,0.184,0.489,0.459,157
1,2B,greedy,143,143,0.225,0.175,0.476,0.477,130
2,31B,stochastic,178,504,0.227,0.224,0.451,0.525,158
3,31B,greedy,167,167,0.224,0.228,0.431,0.517,149
4,4B,stochastic,189,539,0.227,0.219,0.425,0.443,168
5,4B,greedy,173,173,0.225,0.208,0.468,0.468,158


CV results (top-1, mean +/- std over 5 folds x 3 seeds):


family               A_pt               B_directed               C_combined  \
method                gbm  lasso logreg        gbm  lasso logreg        gbm   
model label_source                                                            
2B    greedy        0.242  0.275  0.296      0.527  0.548  0.540      0.524   
      stochastic    0.268  0.287  0.287      0.556  0.534  0.534      0.540   
31B   greedy        0.220  0.281  0.282      0.409  0.411  0.415      0.389   
      stochastic    0.208  0.278  0.278      0.476  0.482  0.483      0.453   
4B    greedy        0.362  0.375  0.375      0.522  0.520  0.514      0.541   
      stochastic    0.264  0.301  0.301      0.499  0.505  0.505      0.503   

family                           D_deductive               S_social         \
method              lasso logreg         gbm  lasso logreg      gbm  lasso   
model label_source                                                           
2B    greedy        0.530  0.521       0.399  0.399  0.396    0.413  0.473   
      stochastic    0.543  0.543       0.378  0.381  0.381    0.428  0.471   
31B   greedy        0.417  0.417       0.247  0.269  0.266    0.391  0.395   
      stochastic    0.464  0.463       0.316  0.333  0.333    0.400  0.428   
4B    greedy        0.537  0.543       0.375  0.387  0.387    0.484  0.434   
      stochastic    0.484  0.485       0.395  0.380  0.380    0.386  0.384   

family                     
method             logreg  
model label_source         
2B    greedy        0.477  
      stochastic    0.471  
31B   greedy        0.395  
      stochastic    0.428  
4B    greedy        0.435  
      stochastic    0.384

## Held-out permutation importance for the best configuration

For each (model, label source): the best (family, method) by CV mean is
re-evaluated with per-fold permutation of each feature on the held-out games;
the reported importance is the mean top-1 drop. Standardized full-data logreg
coefficients are shown for reference only.


In [10]:
best_rows = []
for (model, label_source), train_df in train_cache.items():
    sub = cv_results[(cv_results["model"] == model) & (cv_results["label_source"] == label_source)]
    if sub.empty:
        continue
    best = sub.loc[sub["top1_mean"].idxmax()]
    cols = FAMILIES[best["family"]]
    _, perm = cv_evaluate(train_df, cols, collect_perm=True)
    for f, delta in perm[best["method"]].items():
        perm_rows.append({"model": model, "label_source": label_source,
                          "family": best["family"], "method": best["method"],
                          "feature": f, "top1_drop": round(delta, 4)})
    best_rows.append({"model": model, "label_source": label_source,
                      "best_family": best["family"], "best_method": best["method"],
                      "top1_mean": best["top1_mean"]})

print("Best configuration per (model, label source):")
display(pd.DataFrame(best_rows))
perm_df = pd.DataFrame(perm_rows)
if not perm_df.empty:
    print("Held-out permutation importance (mean top-1 drop when feature is shuffled):")
    display(perm_df.pivot_table(index="feature", columns=["model", "label_source"],
                                values="top1_drop").round(3))


Best configuration per (model, label source):


,model,label_source,best_family,best_method,top1_mean
0,2B,stochastic,B_directed,gbm,0.556
1,2B,greedy,B_directed,lasso,0.548
2,31B,stochastic,B_directed,logreg,0.483
3,31B,greedy,C_combined,logreg,0.417
4,4B,stochastic,B_directed,logreg,0.505
5,4B,greedy,C_combined,logreg,0.543


Held-out permutation importance (mean top-1 drop when feature is shuffled):


model                             2B               31B                4B  \
label_source                  greedy stochastic greedy stochastic greedy   
feature                                                                    
claims_info_role               0.007      0.005  0.004     -0.007  0.008   
claims_werewolf                0.170      0.181  0.042      0.071  0.160   
deception_count               -0.012      0.022  0.016     -0.003 -0.012   
is_in_role_conflict           -0.007      0.020  0.010      0.008  0.002   
is_self_contradiction          0.002      0.005 -0.002      0.025  0.039   
made_any_claim                 0.040      0.008  0.044      0.059  0.004   
n_distinct_roles_claimed_self  0.009      0.010  0.068      0.084  0.033   
n_utterances                     NaN        NaN  0.002        NaN  0.004   
pt_accusation                    NaN        NaN -0.012        NaN  0.029   
pt_call_for_action               NaN        NaN  0.020        NaN  0.008   
pt_defense                       NaN        NaN  0.008        NaN  0.015   
pt_evidence                      NaN        NaN -0.000        NaN  0.004   
pt_identity_declaration          NaN        NaN -0.012        NaN  0.021   
pt_interrogation                 NaN        NaN -0.002        NaN  0.048   
werewolf_count                 0.180      0.192  0.140      0.168  0.141   

model                                     
label_source                  stochastic  
feature                                   
claims_info_role                   0.006  
claims_werewolf                    0.180  
deception_count                    0.021  
is_in_role_conflict                0.004  
is_self_contradiction             -0.004  
made_any_claim                     0.020  
n_distinct_roles_claimed_self      0.024  
n_utterances                         NaN  
pt_accusation                        NaN  
pt_call_for_action                   NaN  
pt_defense                           NaN  
pt_evidence                          NaN  
pt_identity_declaration              NaN  
pt_interrogation                     NaN  
werewolf_count                     0.101

In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(OUTPUT_DIR / "surrogate_cv_results.csv", index=False)
baselines.to_csv(OUTPUT_DIR / "surrogate_baselines.csv", index=False)
if not perm_df.empty:
    perm_df.to_csv(OUTPUT_DIR / "surrogate_permutation_importance.csv", index=False)
print("saved tables ->", OUTPUT_DIR.relative_to(REPO_ROOT))


saved tables -> analysis\cross_model\voting\prompt_v4\predictive\tables


## Reading guide and caveats

- A model beating **crowd-modal** would mean dialogue features predict the LLM
  vote better than human interpretive consensus does; matching it means LLMs
  and humans read the same surface signals. Neither is causal.
- Family comparisons (A vs B vs C) are the RQ-relevant contrast: whether the
  vote is better explained by *who persuades* (A) or by *what is said about
  whom* (B).
- n is 191 games at best; every mean travels with its fold std, and family
  differences within ~1 std should not be narrated as real.
- Permutation importances are on held-out games; a feature at ~0 adds nothing
  out-of-sample even if its coefficient looks large in a full-data fit.
